# EP01 — Deep Q-Network (DQN)
**COMPSCI 713 · S1 2024 Q1 · 12 marks**

| Part | Topic | Marks |
|------|-------|-------|
| a | Name and describe the two networks | 6 |
| b | Bootstrapping — concept and advantages | 6 |

---

## Part a — The Two Networks [6 marks]

DQN requires **two networks** with identical architecture but different roles:

| Network | Also called | Updated how | Role |
|---------|------------|-------------|------|
| **Online Network** | Q-network | Every training step | Selects actions; its weights are optimised by gradient descent |
| **Target Network** | Frozen network | Every N steps (hard copy) or slowly (soft update) | Provides stable TD targets for the loss calculation |

**Why two?**
If you use a single network for both predicting Q-values *and* providing the TD target, the target shifts every update — like chasing a moving goalpost. Training becomes unstable and often diverges.

The target network is a **lagged copy** of the online network, updated only periodically, which provides stable regression targets and breaks the dangerous feedback loop.

---

## Part b — Bootstrapping in RL [6 marks]

**What is bootstrapping?**

Bootstrapping = updating a value estimate using **another estimate** (not the true final return).

In Q-learning the Bellman update is:

```
Q(s,a) <- Q(s,a) + alpha * [r + gamma * max_a' Q(s',a')  -  Q(s,a)]
                             |___________ TD target _____________|
```

The TD target `r + gamma * max Q(s',a')` uses the current Q-estimate for the next state — not the true discounted return. This is the bootstrap.

**Advantages over non-bootstrapping (Monte Carlo) methods:**

| | Bootstrapping (TD) | Non-bootstrapping (MC) |
|-|-------------------|------------------------|
| **Update timing** | After every step (online) | Must wait for episode end |
| **Variance** | Low — no need to sum many future rewards | High — full return is noisy |
| **Bias** | Some (estimate used in target) | None (true return used) |
| **Environments** | Works in continuing tasks | Requires episodic tasks |
| **Speed** | Fast credit assignment | Slow propagation of signal |

**Bias-Variance trade-off:** TD methods have lower variance but introduce bias; MC has zero bias but high variance. TD typically converges faster in practice.

---

## Code Demo — DQN with Two Networks

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import numpy as np, random
from collections import deque

# ─── Shared architecture for both Online and Target networks ──────────────────
class QNet(nn.Module):
    def __init__(self, s_dim, a_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(s_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, a_dim)  # one Q-value per action
        )
    def forward(self, x): return self.net(x)


class DQNAgent:
    def __init__(self, s_dim=4, a_dim=2, gamma=0.99, lr=1e-3,
                 batch=64, target_update=100):
        self.a_dim, self.gamma, self.batch = a_dim, gamma, batch
        self.target_update, self.steps = target_update, 0

        # ── THE TWO NETWORKS ─────────────────────────────────────────────────
        self.online = QNet(s_dim, a_dim)          # updated every step
        self.target = QNet(s_dim, a_dim)          # frozen copy
        self.target.load_state_dict(self.online.state_dict())
        self.target.eval()                         # never calls .backward()

        self.opt = optim.Adam(self.online.parameters(), lr=lr)
        self.mem = deque(maxlen=10_000)

    def act(self, s, eps=0.1):
        if random.random() < eps: return random.randrange(self.a_dim)
        with torch.no_grad():
            return self.online(torch.FloatTensor(s).unsqueeze(0)).argmax().item()

    def push(self, *transition): self.mem.append(transition)

    def learn(self):
        if len(self.mem) < self.batch: return
        s, a, r, s2, d = zip(*random.sample(self.mem, self.batch))
        S  = torch.FloatTensor(np.array(s))
        A  = torch.LongTensor(a).unsqueeze(1)
        R  = torch.FloatTensor(r).unsqueeze(1)
        S2 = torch.FloatTensor(np.array(s2))
        D  = torch.FloatTensor(d).unsqueeze(1)

        # ── BOOTSTRAPPING: use TARGET network for the TD target ───────────────
        with torch.no_grad():
            td_target = R + self.gamma * self.target(S2).max(1, keepdim=True).values * (1-D)

        loss = nn.MSELoss()(self.online(S).gather(1, A), td_target)
        self.opt.zero_grad(); loss.backward(); self.opt.step()

        # ── HARD UPDATE: copy online → target every C steps ───────────────────
        self.steps += 1
        if self.steps % self.target_update == 0:
            self.target.load_state_dict(self.online.state_dict())
            print(f'  Step {self.steps:4d}: Target network synced')


# Quick smoke test
agent = DQNAgent()
for i in range(300):
    s  = np.random.randn(4).astype('f')
    a  = agent.act(s)
    s2 = np.random.randn(4).astype('f')
    agent.push(s, a, np.random.random(), s2, False)
    agent.learn()
print('DQN smoke test passed.')